# PANINI course project: student-like solution run

**Instructor-only notebook. Do not copy this file into the student release.**

This notebook follows the project story from packaged GSW memory through network inspection, retrieval, RICR, and answer generation. `smoke` mode is CPU-only and executes in a few minutes. `full` mode uses the required Qwen models in 4-bit mode and writes restartable, question-sharded JSONL files.

In [ ]:
# Change only these controls. Use 10-question shards on free Colab.
RUN_MODE = 'smoke'          # 'smoke' or 'full'
DATASET = 'musique'         # '2wiki' or 'musique'
SHARD_START = 0
SHARD_SIZE = 10
USE_GOOGLE_DRIVE = False    # strongly recommended for RUN_MODE='full'
BEAM_WIDTH = 5
CANDIDATES_PER_HOP = 15
RETRIEVAL_POOL = 60
RERANK_BATCH_SIZE = 2
RERANK_MAX_LENGTH = 512
SEED = 232

In [ ]:
from pathlib import Path
import gc, json, os, random, re, subprocess, sys, time

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    REPO_ROOT = Path('/content/panini-course-project')
    if not (REPO_ROOT / 'manifest.json').exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/YigitTurali/panini-course-project.git',
                        str(REPO_ROOT)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(REPO_ROOT / 'requirements-colab.txt')], check=True)
    PACKAGE_ROOT = (REPO_ROOT if DATASET == '2wiki' else
                    REPO_ROOT / 'packages/panini_musique_100')
    if USE_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        CACHE_ROOT = Path('/content/drive/MyDrive/panini-course-cache') / DATASET
    else:
        CACHE_ROOT = Path('/content/panini-course-cache') / DATASET
else:
    SOURCE_ROOT = Path.cwd().resolve()
    while SOURCE_ROOT != SOURCE_ROOT.parent and not (SOURCE_ROOT / 'course_project').exists():
        SOURCE_ROOT = SOURCE_ROOT.parent
    sys.path.insert(0, str(SOURCE_ROOT / 'course_project/src'))
    package_name = 'panini_2wiki_100' if DATASET == '2wiki' else 'panini_musique_100'
    PACKAGE_ROOT = SOURCE_ROOT / 'course_project/release' / package_name
    CACHE_ROOT = SOURCE_ROOT / 'course_project/instructor/solution_cache' / DATASET

CACHE_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
print({'package': str(PACKAGE_ROOT), 'cache': str(CACHE_ROOT), 'mode': RUN_MODE})

In [ ]:
import numpy as np
import pandas as pd
import torch

def release_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_report():
    if not torch.cuda.is_available():
        return {'cuda': False}
    properties = torch.cuda.get_device_properties(0)
    return {
        'cuda': True,
        'gpu': properties.name,
        'total_GiB': round(properties.total_memory / 2**30, 2),
        'allocated_GiB': round(torch.cuda.memory_allocated() / 2**30, 2),
    }

DTYPE = ('bfloat16' if torch.cuda.is_available() and
         torch.cuda.get_device_capability(0)[0] >= 8 else 'float16')
print(gpu_report(), 'dtype=', DTYPE)
if RUN_MODE == 'full' and not torch.cuda.is_available():
    raise RuntimeError('Full mode requires Runtime > Change runtime type > GPU.')

## 1. Audit the memory before using it

A student solution should establish stable-ID alignment and label separation before reporting retrieval results.

In [ ]:
from panini_course import CoursePackage

package = CoursePackage(PACKAGE_ROOT)
public_questions = package.questions('public')
heldout_questions = package.questions('held_out')
entities = package.entities()
qa_rows = package.qa_pairs()
qa_by_id = {row['qa_uid']: row for row in qa_rows}

entity_ids = json.loads((PACKAGE_ROOT / 'embeddings/entity_ids.json').read_text())
qa_ids = json.loads((PACKAGE_ROOT / 'embeddings/qa_ids.json').read_text())
entity_matrix = np.load(PACKAGE_ROOT / 'embeddings/entity_embeddings.npy', mmap_mode='r')
qa_matrix = np.load(PACKAGE_ROOT / 'embeddings/qa_embeddings.npy', mmap_mode='r')
assert len(entity_ids) == entity_matrix.shape[0] == len(set(entity_ids))
assert len(qa_ids) == qa_matrix.shape[0] == len(set(qa_ids))
forbidden = {'answer', 'answer_aliases', 'supporting_facts', 'evidences'}
assert all(not (forbidden & set(row)) for row in heldout_questions)
pd.DataFrame([{'split': 'development', 'questions': len(public_questions)},
              {'split': 'held_out', 'questions': len(heldout_questions)},
              {'split': 'corpus', 'entities': len(entities), 'qa_pairs': len(qa_rows)}])

In [ ]:
import networkx as nx
from panini_course.graph import (build_entity_projection, build_native_gsw_graph,
                                  build_unreconciled_entity_projection)

native = build_native_gsw_graph(package.gsw_paths())
unreconciled = build_unreconciled_entity_projection(native)
surface = build_entity_projection(native)

def graph_summary(name, graph):
    simple = nx.Graph(graph.to_undirected()) if graph.is_directed() else nx.Graph(graph)
    components = sorted((len(c) for c in nx.connected_components(simple)), reverse=True)
    return {'graph': name, 'nodes': simple.number_of_nodes(),
            'edges': simple.number_of_edges(), 'components': len(components),
            'giant': components[0], 'giant_fraction': components[0] / simple.number_of_nodes(),
            'isolates': nx.number_of_isolates(simple)}

network_table = pd.DataFrame([graph_summary('native', native),
                              graph_summary('unreconciled', unreconciled),
                              graph_summary('exact_surface', surface)])
network_table

**Student-style interpretation.** The unreconciled projection measures the structure explicitly present in document-local GSWs. The exact-surface graph measures that structure *plus* the consequences of an identity rule. If its giant component changes sharply, I cannot call the corpus globally connected without auditing the merged names. Generic values such as years, nationalities, and occupations may become artificial bridges. I therefore use reconciliation only for the network-sensitivity questions; retrieval below always maps an entity hit back to its originating document-local node.

## 2. Establish sparse and dense retrieval baselines

The original question has a supplied query vector. Answer-instantiated questions later in RICR may require the runtime encoder.

In [ ]:
from panini_course import BM25Index, DenseIndex, QueryEmbeddingStore, TfidfIndex

entity_bm25 = BM25Index.load(PACKAGE_ROOT / 'indices/entity_bm25.joblib',
                                 PACKAGE_ROOT / 'indices/entity_ids.json', source='entity_bm25')
qa_bm25 = BM25Index.load(PACKAGE_ROOT / 'indices/qa_bm25.joblib',
                             PACKAGE_ROOT / 'indices/qa_ids.json', source='qa_bm25')
qa_tfidf = TfidfIndex.load(PACKAGE_ROOT / 'indices/qa_tfidf.npz',
                                PACKAGE_ROOT / 'indices/qa_tfidf_vectorizer.joblib',
                                PACKAGE_ROOT / 'indices/qa_ids.json', source='qa_tfidf')
qa_dense = DenseIndex.load(PACKAGE_ROOT / 'indices/qa_qwen3_8b_ip.faiss',
                               PACKAGE_ROOT / 'indices/qa_ids.json', source='qa_dense')
query_store = QueryEmbeddingStore.load(PACKAGE_ROOT / 'embeddings/query_embeddings.npy',
                                        PACKAGE_ROOT / 'embeddings/query_ids.json',
                                        PACKAGE_ROOT / 'embeddings/queries.jsonl')

example = public_questions[0]
query = example['question']
rankings = {
    'tfidf': qa_tfidf.search(query, 5),
    'bm25': qa_bm25.search(query, 5),
    'dense': qa_dense.search(query_store.get(query), 5),
}
rows = []
for method, hits in rankings.items():
    for hit in hits:
        row = qa_by_id[hit.item_id]
        rows.append({'method': method, 'rank': hit.rank, 'score': hit.score,
                     'stored_question': row['question'],
                     'answer': '; '.join(row['answer_names'])})
pd.DataFrame(rows)

## 3. Implement and test the RICR core

These functions are written in the notebook to mirror the work expected from a student. The CPU smoke run uses the real packaged BM25 indices and a reviewed public decomposition.

In [ ]:
from panini_course import Candidate
from panini_course.ricr import ChainState, geometric_mean, instantiate_question

def prune_unique_answers_solution(chains, beam_width):
    ordered = sorted(chains, key=lambda chain: (
        -chain.score, tuple(step.qa_uid for step in chain.steps)))
    kept, seen = [], set()
    for chain in ordered:
        key = ' '.join(re.findall(r'\w+', chain.current_answer.casefold()))
        if key in seen:
            continue
        seen.add(key)
        kept.append(chain)
        if len(kept) == beam_width:
            break
    return kept

def run_linear_ricr_solution(plan, retrieve, beam_width=5, candidates_per_hop=15):
    steps = [step for step in plan if step.get('requires_retrieval', True)]
    if not steps:
        return []
    first = instantiate_question(steps[0]['question'], {})
    beams = prune_unique_answers_solution([
        ChainState((candidate,), {1: candidate.answer}, max(candidate.score, 1e-12))
        for candidate in retrieve(first, candidates_per_hop)], beam_width)
    for local_step, template in enumerate(steps[1:], start=2):
        expansions = []
        for beam in beams:
            question = instantiate_question(template['question'], beam.answers_by_step)
            for candidate in retrieve(question, candidates_per_hop):
                evidence = beam.steps + (candidate,)
                answers = dict(beam.answers_by_step)
                answers[local_step] = candidate.answer
                expansions.append(ChainState(evidence, answers,
                    geometric_mean([item.score for item in evidence])))
        beams = prune_unique_answers_solution(expansions, beam_width)
        if not beams:
            break
    return beams

In [ ]:
from panini_course import DualRetriever
from panini_course.metrics import exact_match

sparse_dual = DualRetriever(entity_index=entity_bm25, qa_index=qa_bm25,
                             entity_rows=entities, qa_rows=qa_rows)
def sparse_candidates(text, top_k):
    hits = sparse_dual.search(text, entity_top_k=20, qa_top_k=20,
                              fused_top_k=RETRIEVAL_POOL)
    candidates = []
    for hit in hits:
        row = qa_by_id[hit.item_id]
        for answer in row['answer_names']:
            candidates.append(Candidate(row['qa_uid'], answer, hit.score, row['question']))
    return candidates[:top_k]

reviewed = package.decompositions()
smoke_question = next(q for q in public_questions if q['question_id'] in reviewed)
smoke_plan = reviewed[smoke_question['question_id']]
smoke_beams = run_linear_ricr_solution(smoke_plan, sparse_candidates,
                                       BEAM_WIDTH, CANDIDATES_PER_HOP)
smoke_table = pd.DataFrame([{
    'chain_score': beam.score, 'current_answer': beam.current_answer,
    'qa_path': ' -> '.join(step.qa_uid for step in beam.steps),
    'questions': ' | '.join(step.question for step in beam.steps)
} for beam in smoke_beams])
beam_recovery = max(exact_match(beam.current_answer,
                                 [smoke_question['answer'], *smoke_question.get('answer_aliases', [])])
                    for beam in smoke_beams)
print({'question': smoke_question['question'], 'gold': smoke_question['answer'],
       'complete_answer_recovered_in_beam': beam_recovery})
assert beam_recovery == 1.0
smoke_table

## 4. Full GPU stage A: decomposition

Run full mode in 10-question shards. Each record is appended immediately, so a terminated Colab session loses at most the question currently being processed. The next run skips completed IDs.

In [ ]:
questions_all = public_questions + heldout_questions
question_shard = questions_all[SHARD_START:SHARD_START + SHARD_SIZE]
plan_path = CACHE_ROOT / f'plans_{SHARD_START:03d}_{SHARD_START + len(question_shard):03d}.jsonl'

if RUN_MODE == 'full':
    from panini_course.qwen_models import QwenDecomposer
    existing = {}
    if plan_path.exists():
        existing = {row['question_id']: row for row in
                    (json.loads(line) for line in plan_path.open() if line.strip())}
    decomposer = QwenDecomposer('yigitturali/GSW-QA-Decomposer-Qwen3-4B',
        PACKAGE_ROOT / 'models/decomposition_prompt.txt', quantized=True,
        dtype=DTYPE, device_map='auto')
    for row in question_shard:
        if row['question_id'] in existing:
            continue
        started = time.perf_counter()
        try:
            plan = decomposer.decompose(row['question'], max_new_tokens=768)
            record = {'question_id': row['question_id'], 'question': row['question'],
                      'decomposed_questions': plan,
                      'seconds': time.perf_counter() - started}
        except Exception as error:
            record = {'question_id': row['question_id'], 'question': row['question'],
                      'error': repr(error), 'seconds': time.perf_counter() - started}
        with plan_path.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    del decomposer
    release_gpu()
else:
    print('Smoke mode: using the packaged reviewed decomposition instead of loading a model.')

## 5. Full GPU stage B: dual retrieval, reranking, and RICR

The helper below splits common parallel decompositions into independent linear branches, remaps placeholder numbers within each branch, runs RICR, and merges evidence by stable QA ID.

In [ ]:
PLACEHOLDER = re.compile(r'<ENTITY_Q(\d+)>')
def linear_retrieval_sequences(plan):
    retrieval = {index: step for index, step in enumerate(plan, start=1)
                 if step.get('requires_retrieval', True)}
    parents = {index: [int(x) for x in PLACEHOLDER.findall(step['question'])
                       if int(x) in retrieval]
               for index, step in retrieval.items()}
    children = {index: set() for index in retrieval}
    for child, parent_ids in parents.items():
        for parent in parent_ids:
            children[parent].add(child)
    sinks = [index for index in retrieval if not children[index]]
    sequences = []
    for sink in sinks:
        path, current = [], sink
        while True:
            path.append(current)
            if not parents[current]:
                break
            if len(parents[current]) != 1:
                raise ValueError('A retrieval node with multiple retrieval parents needs an explicit join.')
            current = parents[current][0]
        path.reverse()
        remap = {original: local for local, original in enumerate(path, start=1)}
        local_plan = []
        for original in path:
            text = PLACEHOLDER.sub(lambda match: f'<ENTITY_Q{remap[int(match.group(1))]}>',
                                   retrieval[original]['question'])
            local_plan.append({'question': text, 'requires_retrieval': True})
        sequences.append(local_plan)
    return sequences

assert len(linear_retrieval_sequences(smoke_plan)) >= 1

In [ ]:
evidence_path = CACHE_ROOT / f'evidence_{SHARD_START:03d}_{SHARD_START + len(question_shard):03d}.jsonl'
if RUN_MODE == 'full':
    from panini_course.qwen_models import QwenQueryEncoder, QwenReranker
    if gpu_report()['total_GiB'] < 14:
        raise RuntimeError('This stage needs approximately 11 GiB free GPU memory; use a 15–16 GiB runtime.')
    plans = {row['question_id']: row for row in
             (json.loads(line) for line in plan_path.open() if line.strip())}
    completed = set()
    if evidence_path.exists():
        completed = {json.loads(line)['question_id'] for line in evidence_path.open() if line.strip()}
    encoder = QwenQueryEncoder(quantized=True, dtype=DTYPE, device_map='auto')
    reranker = QwenReranker(quantized=True, dtype=DTYPE, device_map='auto',
                              max_length=RERANK_MAX_LENGTH)
    neural_dual = DualRetriever(entity_index=entity_bm25, qa_index=qa_dense,
                                entity_rows=entities, qa_rows=qa_rows)
    vector_cache = {}
    def query_vector(text):
        if text not in vector_cache:
            try:
                vector_cache[text] = query_store.get(text)
            except KeyError:
                vector_cache[text] = encoder.encode([text], max_length=256)[0]
        return vector_cache[text]
    def neural_candidates(text, top_k):
        hits = neural_dual.search(text, query_vector=query_vector(text),
                                  entity_top_k=20, qa_top_k=20,
                                  fused_top_k=RETRIEVAL_POOL)
        rows = [qa_by_id[hit.item_id] for hit in hits]
        documents = [row['search_text'] for row in rows]
        rerank_scores = reranker.score(text, documents, batch_size=RERANK_BATCH_SIZE)
        ranked = sorted(zip(hits, rows, rerank_scores),
                        key=lambda item: (-(0.5 / item[0].rank + 0.5 * item[2]),
                                          item[1]['qa_uid']))
        candidates = []
        for hit, row, rerank_score in ranked:
            score = 0.5 / hit.rank + 0.5 * rerank_score
            for answer in row['answer_names']:
                candidates.append(Candidate(row['qa_uid'], answer, float(score),
                                            row['question'], {'reranker': rerank_score}))
        return candidates[:top_k]
    for question_row in question_shard:
        qid = question_row['question_id']
        if qid in completed or qid not in plans or 'error' in plans[qid]:
            continue
        started = time.perf_counter()
        branch_beams = [run_linear_ricr_solution(sequence, neural_candidates,
                         BEAM_WIDTH, CANDIDATES_PER_HOP)
                        for sequence in linear_retrieval_sequences(plans[qid]['decomposed_questions'])]
        evidence = {}
        for beams in branch_beams:
            for beam in beams:
                for candidate in beam.steps:
                    old = evidence.get(candidate.qa_uid)
                    if old is None or candidate.score > old['score']:
                        evidence[candidate.qa_uid] = {'qa_uid': candidate.qa_uid,
                            'question': candidate.question, 'answer': candidate.answer,
                            'score': candidate.score}
        record = {'question_id': qid, 'question': question_row['question'],
                  'decomposed_questions': plans[qid]['decomposed_questions'],
                  'evidence': sorted(evidence.values(), key=lambda row: -row['score']),
                  'seconds': time.perf_counter() - started}
        with evidence_path.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    del encoder, reranker
    release_gpu()
else:
    print('Smoke mode: neural retrieval/reranking stage skipped.')

## 6. Full GPU stage C: evidence-grounded answers and evaluation

The encoder and reranker are deleted before the 4B answer model is loaded. The answer model receives only deduplicated GSW QA evidence.

In [ ]:
prediction_path = CACHE_ROOT / f'predictions_{SHARD_START:03d}_{SHARD_START + len(question_shard):03d}.jsonl'
if RUN_MODE == 'full':
    from panini_course.qwen_models import QwenAnswerer
    from panini_course.metrics import token_f1
    evidence_rows = [json.loads(line) for line in evidence_path.open() if line.strip()]
    completed = set()
    if prediction_path.exists():
        completed = {json.loads(line)['question_id'] for line in prediction_path.open() if line.strip()}
    answerer = QwenAnswerer(quantized=True, dtype=DTYPE, device_map='auto')
    gold_by_id = {row['question_id']: row for row in public_questions}
    for row in evidence_rows:
        if row['question_id'] in completed:
            continue
        evidence_text = [f"Q: {item['question']} A: {item['answer']}"
                         for item in row['evidence']]
        prediction = answerer.answer(row['question'], evidence_text)
        record = {**row, 'predicted_answer': prediction}
        if row['question_id'] in gold_by_id:
            gold = gold_by_id[row['question_id']]
            aliases = [gold['answer'], *gold.get('answer_aliases', [])]
            record.update({'gold_answer': gold['answer'],
                           'exact_match': exact_match(prediction, aliases),
                           'token_f1': token_f1(prediction, aliases)})
        with prediction_path.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    del answerer
    release_gpu()
    scored = [json.loads(line) for line in prediction_path.open() if line.strip()]
    scored = [row for row in scored if 'exact_match' in row]
    display(pd.DataFrame(scored)[['question_id', 'gold_answer', 'predicted_answer',
                                   'exact_match', 'token_f1', 'seconds']])
    print({'questions': len(scored),
           'EM': np.mean([row['exact_match'] for row in scored]) if scored else None,
           'F1': np.mean([row['token_f1'] for row in scored]) if scored else None})
else:
    print('Smoke run complete. Set RUN_MODE=full and use a GPU to execute stages A–C.')

## Free-tier conclusion

The required model combination is memory-feasible on a 15–16 GiB GPU when quantized: our one-visible-GPU measurement was **3.0 GiB** peak for the 4B decomposer and **11.0 GiB** peak for the 8B encoder plus 8B reranker at batch size 2 and 512 tokens. A previously validated two-hop example required about 12 seconds for decomposition and 10 seconds for retrieval/RICR on faster local GPUs; a free-tier GPU will be slower.

The complete 200-question experiment is therefore feasible as cached shards, not as a promised single session. Google states that free Colab GPU types and limits vary, resources are not guaranteed, and free notebooks run for at most 12 hours depending on availability: https://research.google.com/colaboratory/faq.html. Run 10 questions at a time, persist JSONL files to Drive, and restart between decomposition, retrieval/reranking, and answer generation. The Qwen model cards confirm that the required embedding and reranking checkpoints are 8B models: https://huggingface.co/Qwen/Qwen3-Embedding-8B and https://huggingface.co/Qwen/Qwen3-Reranker-8B.